# Repairing ACAS Xu output properties

ACAS Xu chooses the advisory with the **lowest output score**. Repair minimizes advisory cross-entropy on negated scores while enforcing the imported safe property. VNNLIB normally describes unsafe outputs; CTRAIN complements that formula and preserves strict comparisons and ties.

The default below is a small synthetic CPU fixture, **not a released ACAS network or paper reproduction**. Real ONNX models and VNNLIB files are supplied explicitly. No files are downloaded automatically. [Setup](index.md).

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "case_studies").is_dir() and (p / "pyproject.toml").is_file())
sys.path.insert(0, str(REPO))
WORK = Path(tempfile.mkdtemp(prefix="ctrain-case-study-"))
print("Fresh outputs:", WORK)


## Execute a small repair

The wrapper uses the same property-bound and training paths as a real model. Certification and advisory agreement are reported separately.

In [ ]:
output = WORK / "acasxu"
command = [sys.executable, "-m", "case_studies.acasxu.run",
           "--smoke", "--device", "cpu", "--output", str(output)]
completed = subprocess.run(command, cwd=REPO, check=True, capture_output=True, text=True)
result = json.loads((output / "results.json").read_text())
print(json.dumps(result["metrics"], indent=2))
print("All declared properties certified:", result["after"]["certified"])

## Inspect outcomes

A `certified` cell has a bound proving its property within the floating-point numerical scope. `unknown` is inconclusive; `violated` has a checked counterexample. Advisory agreement is an empirical fidelity metric.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
counts = Counter(cell["outcome"] for cell in result["after"]["cells"])
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(counts), list(counts.values()), color="#3977ad")
ax.set(ylabel="Covering cells", title="Synthetic repair outcomes")
plt.show()

## Use real artifacts and tune repairs

Run from the repository root with local files:

```bash
python -m case_studies.acasxu.run --model data/acasxu/onnx/ACASXU_run2a_2_9_batch_2000.onnx --specification data/acasxu/vnnlib/prop_8.vnnlib --epochs 40 --output runs/case_studies/acasxu_2_9
python -m case_studies.acasxu.hpo --benchmark data/acasxu --network 2_9 --training-method MTL-IBP --trials 20 --output runs/case_studies/acasxu_hpo/2_9
```

Add all other assigned specifications for joint repair. HPO tunes MTL's alpha or SABR's half-width ratio as applicable, along with the shared training parameters. Selection prioritizes fixed-cover certification before fidelity; only the selected checkpoint receives the final full-domain verifier check. Good sampled fidelity does not imply certification, and certification does not imply fidelity.

[Artifact acquisition, benchmark suites and full protocol](https://github.com/ADA-research/CTRAIN/blob/dev/case_studies/acasxu/README.md).